# Train HGRT-10M on WikiText-103

In Colab, choose **Runtime → Change runtime type → A100 GPU**, then **Runtime → Run all**. The model definition is included. A fresh run needs internet access for the corpus and any missing packages. The default starts C1-S8 from scratch. Set `INITIAL_WEIGHTS` to an existing C1-S8 checkpoint to continue from learned weights, or `RESUME_FROM` to restore this notebook’s optimizer and data sampler.

**Why this corpus?** WikiText-103 contains human-written Wikipedia articles and official train, validation and test splits. Its sustained passages give this 10M-parameter byte model both local language patterns and factual context to learn. The pinned raw Parquet release is about 315 MB compressed. It is attributed to the WikiText authors and Wikipedia contributors. The dataset card lists CC BY-SA 3.0 and GFDL. [Dataset card](https://huggingface.co/datasets/Salesforce/wikitext) · [Merity et al.](https://arxiv.org/abs/1609.07843)

Allow **20 minutes for the training loop** plus setup, the two hardware benchmarks and final evaluation. Actual bytes trained depend on measured throughput. Results and checkpoints go to a new folder under `My Drive/HGRT-10M/runs` when Drive mounts. The runtime path is printed otherwise. The original associative-recall gates remain a separate experiment.

In [ ]:
#@title Run settings
from pathlib import Path

TRAIN_MINUTES = 20             # Additional training-loop minutes per invocation
CONTEXT = 512                  # UTF-8 bytes per sequence
BATCH_BYTES = 8192             # Automatically reduced on a benchmark OOM
LR = 0.0005
SEED = 42
DEVICE = 'cuda'
TRY_FUSED = True               # Attempt the optional FLA gated-delta kernel
SAVE_TO_DRIVE = True
INITIAL_WEIGHTS = ''           # Optional C1.pt from the Python documentation pilot
RESUME_FROM = ''               # Optional latest.pt from this notebook
LOCAL_PARQUET_DIR = ''         # Optional offline directory with the four official shard names
OUTPUT_ROOT = ''               # Optional explicit output directory parent
MAX_UPDATES = None             # Optional additional update limit for a short diagnostic
VALID_WINDOWS = 128
TEST_WINDOWS = 256
PROBE_PAIRS = 64               # Per task, each with two matched variants
SAMPLE_BYTES = 120
assert DEVICE in ('cpu', 'cuda') and TRAIN_MINUTES > 0 and LR > 0
assert CONTEXT >= 128 and CONTEXT % 64 == 0 and BATCH_BYTES >= max(2048, 2*CONTEXT)
assert VALID_WINDOWS >= 2 and TEST_WINDOWS >= 2 and PROBE_PAIRS > 0
assert MAX_UPDATES is None or MAX_UPDATES >= 1
assert not (INITIAL_WEIGHTS and RESUME_FROM)

## Verify the runtime and choose an output folder

The setup checks a BF16 matrix multiply. It tries the optional fused gated-delta kernel and can use the CUDA PyTorch implementation.

In [ ]:
import contextlib, gc, hashlib, importlib, importlib.metadata, importlib.util, json
import os, shutil, statistics, subprocess, sys, tempfile, time

def pip_install(*requirements, required=True):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                           '--retries', '1', *requirements], check=required).returncode

if importlib.util.find_spec('torch') is None:
    pip_install('torch==2.9.1+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128')
import torch
if DEVICE == 'cuda':
    if shutil.which('nvidia-smi'):
        subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'])
    if not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
        raise RuntimeError('Select an A100 GPU runtime in Colab, reconnect, and run all cells.')
    x = torch.ones((16, 16), device=DEVICE, dtype=torch.bfloat16)
    assert (x @ x)[0, 0].item() == 16
    del x
for module, package in [('numpy', 'numpy'), ('pyarrow', 'pyarrow'), ('huggingface_hub', 'huggingface-hub'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        pip_install(package)
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

backend = 'torch'
if DEVICE == 'cuda' and TRY_FUSED:
    try:
        from fla.ops.gated_delta_rule import chunk_gated_delta_rule
        backend = 'fla'
    except ImportError:
        # Keep the working CUDA Torch/Triton pair while resolving optional dependencies.
        with tempfile.TemporaryDirectory() as tmp:
            pins = Path(tmp)/'constraints.txt'
            versions = {d: importlib.metadata.version(d) for d in ('torch', 'triton')
                        if importlib.util.find_spec(d)}
            pins.write_text(''.join(f'{d}=={v}\n' for d, v in versions.items()))
            if pip_install('-c', str(pins), 'fla-core', required=False) == 0:
                importlib.invalidate_caches()
                try:
                    from fla.ops.gated_delta_rule import chunk_gated_delta_rule
                    backend = 'fla'
                except ImportError as exc:
                    print('Using PyTorch gated delta:', exc)

local_root = Path('/content') if Path('/content').is_dir() else Path.cwd()
parent = Path(OUTPUT_ROOT) if OUTPUT_ROOT else local_root/'HGRT-10M'/'runs'
if SAVE_TO_DRIVE and not OUTPUT_ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        mount = Path('/content/drive/MyDrive')
        if not mount.is_dir():
            raise OSError('MyDrive unavailable')
        parent = mount/'HGRT-10M'/'runs'
    except Exception as exc:
        print(f'Drive unavailable ({type(exc).__name__}). This run will be stored on the runtime disk.')
parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix=time.strftime('wiki-%Y%m%d-%H%M%S-'), dir=parent))
data_cache = tempfile.TemporaryDirectory(prefix='hgrt-wikitext-')
cache = Path(data_cache.name)  # Fast local disk for this session
device = torch.device(DEVICE)
torch.manual_seed(SEED)

def precision(dev):
    return torch.autocast('cuda', dtype=torch.bfloat16) if torch.device(dev).type == 'cuda' else contextlib.nullcontext()

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

def save_json(name, obj):
    path = run_dir/name
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    tmp.replace(path)

def event(**row):
    with (run_dir/'metrics.jsonl').open('a') as f:
        print(json.dumps(row, allow_nan=False), file=f)
    print(row, flush=True)

hardware = dict(device=torch.cuda.get_device_name() if DEVICE == 'cuda' else 'CPU',
                capacity_bytes=torch.cuda.get_device_properties(device).total_memory if DEVICE == 'cuda' else None,
                precision='bf16 autocast' if DEVICE == 'cuda' else 'fp32', torch=str(torch.__version__),
                cuda_build=torch.version.cuda, numpy=np.__version__)
print(hardware, '\nResults:', run_dir)

## Model

This is the project’s eight-layer C1-S8 implementation with 10,013,977 trainable parameters, six gated delta layers, two local attention layers, 64-byte chunks and eight tree slots. Each UTF-8 byte maps directly to one of 256 IDs.

In [ ]:
#@title HGRT-10M model definition
"""Eight-layer causal LM. Training builds binary-tree levels in parallel."""
import math
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)
class Shape:
    width: int = 256
    heads: int = 4
    chunk: int = 64
    slot_dim: int = 128
    vocab: int = 256
    target: int = 10_000_000


def linear(a, b):
    return nn.Linear(a, b, bias=False)


def delta_reference(q, k, v, g, beta, state):
    """Solve the gated delta recurrence in small blocks, with FP32 accumulation."""
    b, length, h, d = q.shape
    block = min(32, length)
    pad = (-length) % block

    def blocks(x):
        x = F.pad(x.float(), (0, 0, 0, 0, 0, pad))
        return x.transpose(1, 2).reshape(b, h, -1, block, x.size(-1))

    with torch.autocast(q.device.type, enabled=False):
        qb, kb, vb = map(blocks, (q, k, v))
        beta = blocks(beta.unsqueeze(-1))
        log_decay = blocks(g.unsqueeze(-1)).cumsum(-2)
        decay = (log_decay - log_decay.transpose(-1, -2)).clamp_max(0).exp().tril()
        prefix = log_decay.exp()
        bk = beta * kb
        matrix = ((bk @ kb.transpose(-1, -2)) * decay).tril(-1)
        matrix = matrix + torch.eye(block, device=q.device)
        rhs = torch.cat((beta * vb, bk * prefix), -1)
        u, w = torch.linalg.solve_triangular(matrix, rhs, upper=False, unitriangular=True).chunk(2, -1)
        scores = (qb @ kb.transpose(-1, -2)) * decay
        end_keys = kb * decay[..., -1, :].unsqueeze(-1)
        if state is None:
            state = q.new_zeros((b, h, d, d), dtype=torch.float32)
        results = []
        for i in range(kb.size(2)):
            updates = u[:, :, i] - w[:, :, i] @ state
            results.append((qb[:, :, i] * prefix[:, :, i]) @ state + scores[:, :, i] @ updates)
            state = prefix[:, :, i, -1:] * state + end_keys[:, :, i].transpose(-1, -2) @ updates
        out = torch.cat(results, 2).transpose(1, 2)[:, :length]
    return out.to(q.dtype), state


class Delta(nn.Module):
    def __init__(self, s, backend):
        super().__init__()
        self.h, self.d = s.heads, s.width // s.heads
        self.backend = backend
        self.qkv = linear(s.width, 3 * s.width)
        self.conv = nn.Conv1d(3*s.width, 3*s.width, 4, groups=3*s.width, bias=False)
        self.gates = linear(s.width, 2*s.heads)
        self.decay_bias = nn.Parameter(torch.linspace(-5, -2, s.heads))
        self.out_gate = linear(s.width, s.width)
        self.out = linear(s.width, s.width)

    def forward(self, x, cache, offset, streaming):
        qkv = self.qkv(x).transpose(1, 2)
        prev, state = cache if cache is not None else (None, None)
        history = F.pad(qkv, (3, 0)) if prev is None else torch.cat((prev, qkv), -1)
        shaped = F.silu(self.conv(history).transpose(1, 2)).reshape(*x.shape[:2], 3, self.h, self.d)
        q, k, v = shaped.unbind(2)
        q, k = (F.normalize(t.float(), dim=-1, eps=1e-6).to(v.dtype) for t in (q, k))
        decay, beta = self.gates(x).float().chunk(2, -1)
        g = -F.softplus(decay + self.decay_bias)
        beta = beta.sigmoid()
        if self.backend == 'fla':
            from fla.ops.gated_delta_rule import chunk_gated_delta_rule
            out, state = chunk_gated_delta_rule(q.contiguous(), k.contiguous(), v.contiguous(),
                g.contiguous(), beta.to(v.dtype).contiguous(), scale=1., initial_state=state,
                output_final_state=streaming)
        else:
            out, state = delta_reference(q, k, v, g, beta, state)
        out = F.rms_norm(out, (self.d,), eps=1e-5).reshape_as(x)
        return self.out(out * F.silu(self.out_gate(x))), ((history[..., -3:].clone(), state) if streaming else None)


class Attention(nn.Module):
    def __init__(self, s, local):
        super().__init__()
        self.h, self.d = s.heads, s.width // s.heads
        self.window = s.chunk if local else 0
        self.qkv = linear(s.width, 3*s.width)
        self.out = linear(s.width, s.width)
        self.register_buffer("freq", 10000 ** (-torch.arange(0, self.d, 2) / self.d), persistent=False)

    def forward(self, x, cache, offset, streaming):
        b, length, width = x.shape
        q, k, v = self.qkv(x).reshape(b, length, 3, self.h, self.d).permute(2, 0, 3, 1, 4)
        angle = torch.arange(offset, offset+length, device=x.device)[:, None] * self.freq
        co, si = angle.cos().to(q.dtype), angle.sin().to(q.dtype)

        def rotate(t):
            a, z = t.chunk(2, -1)
            return torch.cat((a*co - z*si, z*co + a*si), -1)

        q, k = rotate(q), rotate(k)
        if cache is not None:
            k, v = (torch.cat((a, z), 2) for a, z in zip(cache, (k, v)))
        if self.window and not streaming:
            # Each block sees the previous and current block, with an exact sliding mask.
            w = self.window
            pad = (-length) % w
            def split(t):
                return F.pad(t, (0, 0, 0, pad)).reshape(b, self.h, -1, w, self.d).transpose(1, 2)
            def adjacent(t):
                cur = split(t)
                prev = F.pad(cur[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))
                return torch.cat((prev, cur), 3)
            i = torch.arange(w, device=x.device)[:, None] + w
            j = torch.arange(2*w, device=x.device)
            mask = ((i >= j) & (i-j < w)).expand((length+pad)//w, 1, w, 2*w).clone()
            mask[0, :, :, :w] = False
            y = F.scaled_dot_product_attention(split(q), adjacent(k), adjacent(v), attn_mask=mask)
            y = y.transpose(1, 2).reshape(b, self.h, -1, self.d)[:, :, :length]
        elif not self.window and cache is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            difference = torch.arange(length, device=x.device)[:, None] + k.size(2)-length - torch.arange(k.size(2), device=x.device)
            valid = difference >= 0
            if self.window:
                valid &= difference < self.window
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=valid)
        keep = self.window-1 if self.window else k.size(2)
        next_cache = (k[:, :, -keep:].clone(), v[:, :, -keep:].clone()) if streaming else None
        return self.out(y.transpose(1, 2).reshape(b, length, width)), next_cache


class Tree(nn.Module):
    def __init__(self, s, slots):
        super().__init__()
        self.chunk = s.chunk
        self.project = linear(s.width, s.slot_dim)
        self.slots = nn.Parameter(torch.randn(slots, s.slot_dim) * 0.2)
        self.merge_proj = linear(s.slot_dim, s.slot_dim)
        nn.init.eye_(self.merge_proj.weight)
        self.gain = nn.Parameter(torch.tensor(0.1))

    def write(self, source):
        z = self.project(source)
        weights = (self.slots.to(z.dtype) @ z.transpose(-1, -2)) / math.sqrt(z.size(-1))
        return weights.softmax(-1) @ z

    def merge(self, left, right):
        return self.merge_proj((left + right) * .5)

    def read(self, x, nodes, mask=None):
        q = self.project(x)
        a = F.scaled_dot_product_attention(q, nodes, nodes, attn_mask=mask)
        return self.gain * F.linear(a, self.project.weight.T)

    def forward(self, x, source, mode):
        if mode == 'off' or x.size(1) <= self.chunk:
            return x
        if mode == 'shuffle':
            if len(x) < 2:
                raise ValueError("Shuffled memory requires independent donor sequences")
            source = source.roll(1, 0)
        b, length, d = x.shape
        c = self.chunk
        padded = F.pad(x, (0, 0, 0, (-length) % c)).reshape(b, -1, c, d)
        chunks = padded.size(1)
        level = self.write(source[:, :(chunks-1)*c].reshape(b, chunks-1, c, d))
        completed = torch.arange(chunks, device=x.device)
        nodes, masks = [], []
        span = 1
        while True:
            index = completed // span
            nodes.append(level[:, (index-1).clamp(0, level.size(1)-1)])
            masks.append((index % 2 == 1)[:, None].expand(-1, level.size(2)))
            pairs = level.size(1) // 2
            if not pairs:
                break
            level = self.merge(level[:, :2*pairs:2], level[:, 1:2*pairs:2])
            span *= 2
        # Chunk zero has no history. B,C act as batch and head axes in SDPA.
        memory = torch.cat(nodes, 2)[:, 1:]
        mask = torch.cat(masks, 1)[1:, None, :]
        reads = self.read(padded[:, 1:], memory, mask)
        return x + F.pad(reads.flatten(1, 2), (0, 0, c, 0))[:, :length]


class Block(nn.Module):
    def __init__(self, s, mix, hidden):
        super().__init__()
        self.mix = mix
        self.n1 = nn.RMSNorm(s.width, eps=1e-5)
        self.n2 = nn.RMSNorm(s.width, eps=1e-5)
        self.up, self.down = linear(s.width, 2*hidden), linear(hidden, s.width)

    def forward(self, x, cache, offset, streaming):
        y, cache = self.mix(self.n1(x), cache, offset, streaming)
        x = x + y
        gate, val = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(gate)*val), cache


class LM(nn.Module):
    def __init__(self, kind='C1', slots=8, s=Shape(), backend='torch'):
        super().__init__()
        if kind not in ('A0', 'B1', 'C1'):
            raise ValueError(kind)
        self.s, self.kind = s, kind
        self.embed = nn.Embedding(s.vocab, s.width)
        nn.init.normal_(self.embed.weight, std=.02)
        self.norm = nn.RMSNorm(s.width, eps=1e-5)
        mixers = [Attention(s, kind != 'A0') if kind == 'A0' or i in (3, 7)
                  else Delta(s, backend) for i in range(8)]
        # Centre the shared B1/C1 width between the two parameter counts.
        fixed = sum(p.numel() for module in [self.embed, self.norm, *mixers] for p in module.parameters()) + 16*s.width
        tree_mid = (s.width*s.slot_dim + s.slot_dim**2 + 16*s.slot_dim + 1)/2 if kind != 'A0' else 0
        self.hidden = max(4, 4 * round((s.target-fixed-tree_mid)/(8*3*s.width*4)))
        self.blocks = nn.ModuleList(Block(s, m, self.hidden) for m in mixers)
        self.tree = Tree(s, slots) if kind == 'C1' else None

    def forward(self, tokens, mode='on', positions=None):
        x = self.embed(tokens)
        for i, layer in enumerate(self.blocks):
            if i == 7 and self.tree is not None:
                x = self.tree(x, source, mode)
            x, _ = layer(x, None, 0, False)
            if i == 3:
                source = x
        if positions is not None:
            x = x.index_select(1, positions)
        return F.linear(self.norm(x), self.embed.weight)

    def stream_chunk(self, tokens, state=None):
        """Process one complete aligned chunk, or a terminal partial chunk."""
        caches, forest, offset = state or ([None]*8, {}, 0)
        if offset % self.s.chunk or not 0 < tokens.size(1) <= self.s.chunk:
            raise ValueError("stream_chunk expects aligned chunks")
        x = self.embed(tokens)
        new_caches = []
        for i, layer in enumerate(self.blocks):
            if i == 7 and self.tree is not None and forest:
                x = x + self.tree.read(x, torch.cat([forest[j] for j in sorted(forest)], 1))
            x, cache = layer(x, caches[i], offset, True)
            new_caches.append(cache)
            if i == 3:
                source = x
        forest = dict(forest)
        if self.tree is not None and tokens.size(1) == self.s.chunk:
            node = self.tree.write(source)
            level = 0
            while level in forest:
                node = self.tree.merge(forest.pop(level), node)
                level += 1
            forest[level] = node
        return F.linear(self.norm(x), self.embed.weight), (new_caches, forest, offset + tokens.size(1))

## Prepare the corpus

Read the two train shards and the official validation and test shards separately. Each row contributes its UTF-8 text with a terminating newline. Blank rows become a newline. Local memory-mapped byte files keep the full corpus out of GPU memory. Random training windows are drawn with replacement from the train split. Fixed nonoverlapping validation and test windows are selected independently. Split hashes, byte counts and exact offsets are saved. A local Parquet override is labeled with its actual path and hashes. [Pinned upstream revision](https://huggingface.co/datasets/Salesforce/wikitext/commit/b08601e04326c79dfdd32d625aee71d232d685c3)

In [ ]:
REPO = 'Salesforce/wikitext'
REV = 'b08601e04326c79dfdd32d625aee71d232d685c3'
SUBSET = 'wikitext-103-raw-v1'
FILES = {'train': ['train-00000-of-00002.parquet', 'train-00001-of-00002.parquet'],
         'validation': ['validation-00000-of-00001.parquet'], 'test': ['test-00000-of-00001.parquet']}

def make_split(split):
    path = cache/f'{split}.bin'
    digest = hashlib.sha256()
    count = 0
    with path.open('wb') as dst:
        for name in FILES[split]:
            shard = (Path(LOCAL_PARQUET_DIR)/name if LOCAL_PARQUET_DIR else
                     hf_hub_download(REPO, f'{SUBSET}/{name}', repo_type='dataset', revision=REV))
            for batch in pq.ParquetFile(shard).iter_batches(columns=['text'], batch_size=4096):
                lines = batch.column(0).to_pylist()
                raw = ''.join(line if line.endswith('\n') else line+'\n' for line in lines).encode('utf8')
                dst.write(raw)
                digest.update(raw)
                count += len(lines)
    if path.stat().st_size < max(2*CONTEXT+1, 2049):
        raise ValueError(f'{split} split too small for the selected benchmark')
    return (np.memmap(path, dtype=np.uint8, mode='r'),
            dict(bytes=path.stat().st_size, rows=count, sha256=digest.hexdigest(), files=FILES[split]))

corpus, split_info = {}, {}
for split in FILES:
    corpus[split], split_info[split] = make_split(split)
    print(split, split_info[split]['rows'], 'rows,', len(corpus[split]), 'bytes')
data_manifest = dict(source=str(LOCAL_PARQUET_DIR) if LOCAL_PARQUET_DIR else REPO,
                     revision=None if LOCAL_PARQUET_DIR else REV, subset=SUBSET, splits=split_info)
save_json('corpus.json', data_manifest)

def windows(raw, starts, length):
    ids = torch.from_numpy(raw[np.asarray(starts)[:, None]+np.arange(length+1)]).to(device, dtype=torch.long)
    return ids[:, :-1], ids[:, 1:]

def train_batch(rng, length, batch):
    raw = corpus['train']
    return windows(raw, rng.integers(0, len(raw)-length, size=batch), length)

def fixed_starts(raw, n, seed):
    # Disjoint target bytes; randomize batch neighbours for the shuffled-tree control.
    eligible = (len(raw)-1)//CONTEXT
    n = min(n, eligible)//2*2
    if n < 2:
        raise ValueError('Need two non-overlapping evaluation windows')
    return np.random.default_rng(seed).choice(eligible, n, replace=False)*CONTEXT

starts = {s: fixed_starts(corpus[s], n, k) for s,n,k in
          [('validation', VALID_WINDOWS, 3100), ('test', TEST_WINDOWS, 9413)]}
save_json('eval_windows.json', {s:p.tolist() for s,p in starts.items()})

def fit_byte_baselines(raw):
    uni = np.ones(256, dtype=np.int64)
    bi = np.ones(65536, dtype=np.int64)
    for i in range(0, len(raw), 1<<20):
        piece = raw[i:i+(1<<20)]
        uni += np.bincount(piece, minlength=256)
        size = min(len(piece), len(raw)-1-i)
        if size:
            transitions = piece[:size].astype(np.int32)*256 + raw[i+1:i+1+size]
            bi += np.bincount(transitions, minlength=65536)
    bi = bi.reshape(256,256)
    return -np.log2(uni/uni.sum()), -np.log2(bi/bi.sum(1, keepdims=True))

uni_bits, bi_bits = fit_byte_baselines(corpus['train'])

def baseline_scores(split):
    blocks = corpus[split][starts[split][:, None]+np.arange(CONTEXT+1)]
    x, y = blocks[:,:-1], blocks[:,1:]
    return dict(unigram_bpb=float(uni_bits[y].mean()), bigram_bpb=float(bi_bits[x,y].mean()))

## Benchmark the selected hardware

First profile C1-S8 at 2,048 bytes with full memory gradients and a real next-byte objective. Then profile the selected training context. Each measurement uses a disposable model, one warm-up update and three synchronized forward/backward/AdamW updates on a resident batch. Peak memory is PyTorch’s allocated GPU memory during training. Batch-one prefill is the median of five synchronized full-sequence inference calls. Data transfer, compilation warm-up and autoregressive decoding are outside the timed rates.

In [ ]:
def new_model(kernel):
    torch.manual_seed(SEED)
    model = LM('C1', 8, backend=kernel).to(device)
    assert sum(p.numel() for p in model.parameters()) == 10_013_977
    return model

def make_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=.01, fused=(DEVICE == 'cuda'))

def update(model, optimizer, x, y, check_tree=False):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with precision(device):
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1,256).float(), y.reshape(-1))
    loss.backward()
    norms = {}
    if check_tree:
        norms = {name: float(p.grad.norm()) if p.grad is not None else 0.
                 for name,p in model.tree.named_parameters()}
        if not all(math.isfinite(v) and v > 0 for v in norms.values()):
            raise FloatingPointError(f'Missing finite tree gradients: {norms}')
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
    optimizer.step()
    return float(loss.detach()), norms

def benchmark(length, token_budget, kernel):
    model = new_model(kernel)
    opt = make_optimizer(model)
    batch = max(1, token_budget//length)
    x, y = train_batch(np.random.default_rng(800), length, batch)
    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    _, norms = update(model, opt, x, y, check_tree=(length == 2048))
    sync()
    times = []
    for _ in range(3):
        start = time.perf_counter()
        update(model, opt, x, y)
        sync()
        times.append(time.perf_counter()-start)
    peak = torch.cuda.max_memory_allocated() if DEVICE == 'cuda' else None
    with torch.inference_mode(), precision(device):
        model.eval()
        model(x[:1])
        sync()
        latency = []
        for _ in range(5):
            start = time.perf_counter()
            model(x[:1])
            sync()
            latency.append(1000*(time.perf_counter()-start))
    return dict(length=length, batch=batch, backend=kernel, timed_train_steps=len(times),
                median_train_step_seconds=statistics.median(times),
                train_bytes_per_second=x.numel()/statistics.median(times),
                peak_allocated_bytes=peak, batch1_prefill_ms=statistics.median(latency),
                tree_gradient_norms=norms)

batch_budget = BATCH_BYTES
while True:
    retry = None
    try:
        bench = [benchmark(length, batch_budget, backend) for length in dict.fromkeys((2048, CONTEXT))]
    except torch.OutOfMemoryError:
        smaller = max(batch_budget//2, 2048, 2*CONTEXT)
        if smaller >= batch_budget:
            raise
        batch_budget = smaller
        retry = f'Retrying after GPU OOM with {smaller} batch bytes.'
    except Exception as exc:
        if backend != 'fla' or isinstance(exc, (FloatingPointError, AssertionError)):
            raise
        backend = 'torch'
        retry = f'FLA benchmark failed ({type(exc).__name__}: {exc}). Retrying with CUDA PyTorch.'
    if retry is None:
        break
    print(retry)
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
batch_size = batch_budget//CONTEXT
eval_batch = max(2, (batch_size//2)*2)
save_json('benchmark.json', dict(hardware=hardware, profiles=bench))
for profile in bench:
    print(f"{profile['length']:4d} bytes, batch {profile['batch']}: "
          f"{profile['train_bytes_per_second']:,.0f} train bytes/s, "
          f"{profile['batch1_prefill_ms']:.1f} ms batch-one prefill, "
          f"peak allocated bytes {profile['peak_allocated_bytes']}")

## Train the language model

AdamW uses BF16 autocast on GPU, gradient clipping at 1.0 and a linear 50-update learning-rate warm-up. Validation uses identical windows throughout. Every 90 seconds of loop time the notebook checks validation and saves `latest.pt`. It also saves at the final update. Resume adds the configured minutes in a new output folder; matching data hashes, batch size, optimizer settings and seed are required. An inherited checkpoint loaded through `INITIAL_WEIGHTS` starts a fresh optimizer.

In [ ]:
@torch.inference_mode()
def evaluate(model, split, mode='on'):
    model.eval()
    nll = 0.
    total = 0
    for i in range(0, len(starts[split]), eval_batch):
        x, y = windows(corpus[split], starts[split][i:i+eval_batch], CONTEXT)
        with precision(device):
            logits = model(x, mode)
            nll += float(F.cross_entropy(logits.float().reshape(-1,256), y.reshape(-1), reduction='sum'))
        total += y.numel()
    return nll/(total*math.log(2))

config = dict(kind='C1', slots=8, seed=SEED, context=CONTEXT, batch=batch_size, lr=LR,
              tokenizer='utf8-bytes', corpus_hashes={k:v['sha256'] for k,v in split_info.items()})
save_json('config.json', dict(config=config, hardware=hardware, backend=backend, minutes=TRAIN_MINUTES,
                              initial_weights=str(INITIAL_WEIGHTS), resume_from=str(RESUME_FROM)))
model = new_model(backend)
optimizer = make_optimizer(model)
train_rng = np.random.default_rng(SEED)
step, bytes_seen = 0, 0
if RESUME_FROM or INITIAL_WEIGHTS:
    ck = torch.load(RESUME_FROM or INITIAL_WEIGHTS, map_location='cpu', weights_only=True)
    if ck['kind'] != 'C1' or ck['slots'] != 8:
        raise ValueError('Select a C1-S8 checkpoint.')
    model.load_state_dict(ck['state_dict'])
    if RESUME_FROM:
        if ck.get('config') != config:
            raise ValueError('Resume requires matching data, model and optimizer settings. Use INITIAL_WEIGHTS for transfer.')
        optimizer.load_state_dict(ck['optimizer'])
        step, bytes_seen = ck['step'], ck['bytes_seen']
        train_rng.bit_generator.state = json.loads(ck['train_rng'])
    del ck

def save_checkpoint():
    tmp = run_dir/'latest.partial'
    torch.save(dict(kind='C1', slots=8, config=config, state_dict=model.state_dict(),
                    optimizer=optimizer.state_dict(), step=step, bytes_seen=bytes_seen,
                    train_rng=json.dumps(train_rng.bit_generator.state)), tmp)
    tmp.replace(run_dir/'latest.pt')

before_bpb = evaluate(model, 'validation')
valid_base = baseline_scores('validation')
curve = [(step, before_bpb)]
event(stage='initial', step=step, bytes_seen=bytes_seen, validation_bpb=before_bpb, **valid_base)

In [ ]:
initial_step = step
start_clock = last_check = time.perf_counter()
deadline = start_clock + 60*TRAIN_MINUTES
while time.perf_counter() < deadline and (MAX_UPDATES is None or step-initial_step < MAX_UPDATES):
    x, y = train_batch(train_rng, CONTEXT, batch_size)
    for group in optimizer.param_groups:
        group['lr'] = LR*min(1., (step+1)/50)
    loss, _ = update(model, optimizer, x, y)
    step += 1
    bytes_seen += x.numel()
    if time.perf_counter()-last_check >= 90:
        bpb = evaluate(model, 'validation')
        curve.append((step,bpb))
        save_checkpoint()
        event(stage='train', step=step, bytes_seen=bytes_seen, loss=loss, validation_bpb=bpb)
        last_check = time.perf_counter()
sync()
train_seconds = time.perf_counter()-start_clock
save_checkpoint()
event(stage='train_done', step=step, bytes_seen=bytes_seen, additional_steps=step-initial_step,
      train_loop_seconds=train_seconds, checkpoint=str(run_dir/'latest.pt'))

## Evaluate held-out text

Bits per byte (BPB) is the average negative log₂ probability of the observed next byte; lower is better. Byte perplexity equals 2ᴮᴾᴮ. Compare models using the same byte targets and context. Train-fitted unigram and bigram baselines score exactly the same test windows. A positive disabled-minus-intact or shuffled-minus-intact difference indicates a benefit for the intact tree on these windows. The shuffled condition rotates completed tree histories across different batch windows. The reserved test subset is intended for the final report; use validation for model choices.

In [ ]:
validation_bpb = evaluate(model,'validation')
test_scores = {mode:evaluate(model, 'test', mode) for mode in ('on','off','shuffle')}
test_base = baseline_scores('test')
curve.append((step, validation_bpb))
lm_results = dict(initial_validation_bpb=before_bpb, final_validation_bpb=validation_bpb,
                  validation_targets=len(starts['validation'])*CONTEXT,
                  test_targets=len(starts['test'])*CONTEXT, test_bpb=test_scores,
                  intact_test_byte_perplexity=2**test_scores['on'], test_baselines=test_base,
                  off_minus_intact_bpb=test_scores['off']-test_scores['on'],
                  shuffle_minus_intact_bpb=test_scores['shuffle']-test_scores['on'],
                  optimizer_steps=step, training_bytes=bytes_seen, additional_steps=step-initial_step,
                  train_loop_seconds=train_seconds)
save_json('lm_results.json', lm_results)
print(json.dumps(lm_results, indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.6))
steps, bpb = zip(*curve)
ax.plot(steps, bpb, color='#18558a', marker='o', markersize=4, label='C1-S8')
ax.axhline(valid_base['unigram_bpb'], color='#666666', linestyle=':', label='Train unigram')
ax.axhline(valid_base['bigram_bpb'], color='#ac5400', linestyle='--', label='Train bigram')
ax.set(xlabel='Optimizer update', ylabel='Held-out bits per byte', title='WikiText validation windows' if not LOCAL_PARQUET_DIR else 'Validation windows')
ax.grid(axis='y', alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir/'validation_curve.png', dpi=150)
plt.show()
plt.close(fig)

## Generate a sample

The sampler carries the model state over completed chunks. The fixed prompt and generator seed make qualitative comparisons easier.

In [ ]:
@torch.inference_mode()
def sample(model, prompt, count, temperature, device):
    tokens = list(prompt.encode('utf8'))
    if not tokens:
        raise ValueError('Supply a nonempty prompt')
    chunk = model.s.chunk
    state = None
    # A full chunk can be committed once. Only recompute the unfinished suffix.
    start = 0
    for _ in range(count):
        while len(tokens)-start > chunk:
            ids = torch.tensor([tokens[start:start+chunk]], device=device)
            _, state = model.stream_chunk(ids,state)
            start += chunk
        ids = torch.tensor([tokens[start:]], device=device)
        logits, pending = model.stream_chunk(ids,state)
        probs = (logits[0,-1].float()/temperature).softmax(0)
        tokens.append(int(torch.multinomial(probs,1).item()))
        if ids.size(1) == chunk:
            state = pending
            start += chunk
    return bytes(tokens).decode('utf8', errors='replace')

model.eval()
torch.manual_seed(123)
prompt = 'The expedition reached the river at dawn. '
with precision(device):
    passage = sample(model, prompt, SAMPLE_BYTES, .8, device)
(run_dir/'sample.txt').write_text(passage)
print(passage)

## Probe factual lookup and two-hop inference

Each item has two matched versions and two equal-byte-length candidate endings. In the direct task, swapping where two containers are located changes the correct place. In the two-hop task, locations stay fixed and swapping which container holds an object changes its inferred location. The LM scores every byte of each candidate including its final period. Templates, fact order and vocabulary choices vary; the probe seed stays fixed across runs. These examples are generated for evaluation only.

For example, suppose a coin is in a box, a key is in a bag, the box is in a barn and the bag is in a shed. The coin is in the barn. Swap which container holds the coin and key, and the answer must change to the shed. **Both-correct pairs** count how often the model gets both versions right. Any context-independent preference for one place gets 50% individual accuracy and zero both-correct pairs. The accuracy intervals resample whole pairs. This probes sensitivity to the supplied facts within these templates.

In [ ]:
def make_probe(n_pairs=64, seed=719):
    """Two counterfactual prompts per item; the correct candidate reverses."""
    import random
    rng = random.Random(seed)
    items = []
    seen = set()
    for task in ('direct', 'two_hop'):
        for pair_id in range(n_pairs):
            while True:
                objects = rng.sample(['coin', 'ring', 'key', 'map', 'pen', 'pin', 'bell', 'comb'], 2)
                boxes = rng.sample(['bag', 'box', 'chest', 'case', 'basket', 'drawer'], 2)
                places = rng.sample(['barn', 'cave', 'shed', 'hall', 'mill', 'yurt', 'shop', 'loft'], 2)
                key = (task, *objects, *boxes, *places)
                if key not in seen:
                    seen.add(key)
                    break
            oi = rng.randrange(2)
            bi = rng.randrange(2)
            # Both queried entities and both fact orders vary independently.
            object_order = rng.sample([0, 1], 2)
            box_order = rng.sample([0, 1], 2)
            template = pair_id % 2
            for variant in (0, 1):
                box_for = [i ^ oi ^ (variant if task == 'two_hop' else 0) for i in (0, 1)]
                place_for = [i ^ bi ^ (variant if task == 'direct' else 0) for i in (0, 1)]
                facts = []
                for i in object_order:
                    if template == 0:
                        facts.append(f"The {objects[i]} is inside the {boxes[box_for[i]]}.")
                    else:
                        facts.append(f"Inside the {boxes[box_for[i]]} is the {objects[i]}.")
                for i in box_order:
                    if template == 0:
                        facts.append(f"The {boxes[i]} is in the {places[place_for[i]]}.")
                    else:
                        facts.append(f"The {places[place_for[i]]} holds the {boxes[i]}.")
                noun = boxes[0] if task == 'direct' else objects[0]
                answer = place_for[0] if task == 'direct' else place_for[box_for[0]]
                prompt = ' '.join(facts) + f" The {noun} is therefore in the"
                items.append(dict(task=task, pair=pair_id, version=variant, template=template,
                                  prompt=prompt, endings=[' '+p+'.' for p in places], answer=answer))
    return items


@torch.inference_mode()
def probe_model(model, items, device, sequence_batch=16):
    model.eval()
    rows = []
    # Each item has two equal-byte-length completions. Teacher-force every
    # completion byte, including the space and period. Right padding is unscored.
    for first in range(0, len(items), max(1, sequence_batch//2)):
        group = items[first:first+max(1, sequence_batch//2)]
        seqs, spans = [], []
        for item in group:
            prefix = item['prompt'].encode('utf8')
            for ending in item['endings']:
                seq = prefix + ending.encode('utf8')
                seqs.append(torch.tensor(list(seq), dtype=torch.long))
                spans.append((len(prefix)-1, len(seq)-1))
        ids = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True).to(device)
        with precision(device):
            logits = model(ids[:, :-1])
        token_nll = F.cross_entropy(logits.float().transpose(1, 2), ids[:, 1:], reduction='none')
        logps = [-float(token_nll[i, a:b].sum()) for i, (a, b) in enumerate(spans)]
        for i, item in enumerate(group):
            a, b = logps[2*i:2*i+2]
            predicted = int(b > a)  # exact ties choose the first of the balanced labels
            rows.append(dict(**item, logp_left=a, logp_right=b, predicted=predicted,
                             correct=predicted == item['answer'], tie=a == b))
    summary = {}
    for task in ('direct', 'two_hop'):
        task_rows = [row for row in rows if row['task'] == task]
        correct = np.array([r['correct'] for r in task_rows]).reshape(-1, 2)
        rng = np.random.default_rng(917)
        resampled = correct.mean(1)[rng.integers(len(correct), size=(2000, len(correct)))].mean(1)
        summary[task] = dict(prompts=correct.size, pairs=len(correct), correct=int(correct.sum()),
                             accuracy=float(correct.mean()), both_correct=int(correct.all(1).sum()),
                             pair_accuracy=float(correct.all(1).mean()),
                             accuracy_ci95=np.quantile(resampled, [.025,.975]).tolist(),
                             ties=sum(r['tie'] for r in task_rows))
    return summary, rows

In [ ]:
probe_summary, probe_rows = probe_model(model, make_probe(PROBE_PAIRS), device, min(32, max(2,batch_size)))
save_json('reasoning.json', dict(seed=719, summary=probe_summary, items=probe_rows))
for task, result in probe_summary.items():
    print(f"{task}: {result['correct']}/{result['prompts']} individual, "
          f"{result['both_correct']}/{result['pairs']} both versions correct")
report = dict(config=config, hardware=hardware, benchmark=bench, language_model=lm_results,
              reasoning=probe_summary, checkpoint=str(run_dir/'latest.pt'))
save_json('report.json', report)
print('Results and checkpoint:', run_dir)

### Reading the result

A fall in held-out BPB measures better prediction on Wikipedia prose. Direct lookup accuracy measures use of an explicitly stated fact. Consistent success on both versions of the two-hop items would support simple compositional inference in this setting. Replication with unseen templates, different relations, longer distances and independent training seeds would be needed to argue for broader reasoning. The prior documentation run’s 1.979 BPB is on a different corpus. Word-level WikiText perplexities also use a different unit.